# Week 7 Midterm Preliminary Research

CST4714 Database Administration

Today you will inspect Metro Support's sample data, read two short PostgreSQL documentation sections, and write a plan for the midterm's index experiment. Work individually. The code is supplied; change only the values marked for you to explore.

**Time:** about 50 minutes. **Writing:** 180–250 words total across the three response cells, plus the small observation table. **Today's submission:** the sharing link to your saved copy of this notebook, with outputs and writing. You can reuse your writing in the final midterm README.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_07/07_midterm_preliminary_research.ipynb)

Open the notebook with the button above. Choose **File > Save a copy in Drive**, then rename your copy **Lastname_Week07_Research**. Connect to a Python 3 runtime and run the code cells from the top. Double-click a text cell to edit your responses. If you downloaded this file, you can also upload it at https://colab.research.google.com/ using **File > Upload notebook**.

This notebook uses Python and embedded synthetic CSV data. It needs no database password, Supabase account, downloaded dataset, or package installation. It does not execute PostgreSQL SQL, enforce its constraints, or measure index performance. Those tests belong in your midterm's PostgreSQL database later.

## The project you are preparing

The individual midterm uses the supplied three-table Metro Support database. Its five parts are: verify data and controls, show a rollback and a commit, explain one blocking incident, test one improvement, and plan recovery after Week 8. The final submission remains **midterm.sql** and **README.md** in one ZIP. Use the full assignment and its rubric for the requirements and Brightspace for the deadline.

Today's research focuses on the recommended improvement: an index for finding the newest open tickets. Other improvements permitted by the assignment remain available, but everyone can use this small example to practice connecting documentation to evidence.

## 1 Inspect the supplied data

Allow about 10 minutes. Run the next cell. The data is copied from the course's Metro Support CSVs; these are the starter values before any midterm changes. Record the three counts in the observation table below.

In [ ]:
import csv
import io
import json
from collections import Counter
from datetime import datetime

# Embedded course data: no network fetch and no database connection.
CSV_DATA = {
    "users": """user_id,display_name,email,role,neighborhood,created_at
101,Requester 101,requester101@example.test,resident,Harbor,2026-01-08T14:20:00Z
102,Requester 102,requester102@example.test,resident,Northside,2026-01-10T09:15:00Z
103,Requester 103,requester103@example.test,resident,Central,2026-01-12T18:05:00Z
104,Requester 104,requester104@example.test,resident,Harbor,2026-01-18T11:40:00Z
201,Agent 201,agent201@example.test,agent,Central,2025-11-03T13:00:00Z
202,Agent 202,agent202@example.test,agent,Northside,2025-11-05T13:00:00Z
203,Agent 203,agent203@example.test,supervisor,Central,2025-09-14T13:00:00Z
204,Sam Okafor,sam.okafor@example.test,analyst,Harbor,2025-12-01T13:00:00Z
""",
    "tickets": """ticket_id,requester_id,assignee_id,category,priority,status,subject,opened_at,closed_at
1001,101,201,streetlight,high,open,Streetlight dark near bus stop,2026-02-01T23:10:00Z,
1002,102,202,sanitation,medium,in_progress,Missed recycling pickup,2026-02-02T15:45:00Z,
1003,103,201,water,urgent,resolved,Low water pressure,2026-02-03T12:05:00Z,2026-02-03T19:40:00Z
1004,104,,parks,low,new,Broken bench slat,2026-02-04T17:20:00Z,
1005,101,202,sanitation,high,resolved,Overflowing corner bin,2026-02-05T14:00:00Z,2026-02-05T20:15:00Z
1006,102,201,streetlight,medium,in_progress,Flickering lamp outside library,2026-02-06T01:30:00Z,
1007,103,202,parks,medium,open,Playground gate will not latch,2026-02-07T16:10:00Z,
1008,104,201,water,high,resolved,Hydrant leaking slowly,2026-02-08T10:25:00Z,2026-02-09T09:05:00Z
1009,101,,transportation,medium,new,Bus shelter panel cracked,2026-02-09T22:15:00Z,
1010,102,202,sanitation,low,closed,Replacement bin request,2026-02-10T13:50:00Z,2026-02-12T16:30:00Z
1011,103,201,transportation,high,open,Crosswalk signal delayed,2026-02-11T08:35:00Z,
1012,104,202,streetlight,low,resolved,Lamp stays on during daytime,2026-02-12T14:45:00Z,2026-02-14T18:10:00Z
""",
    "ticket_events": """event_id,ticket_id,actor_id,event_type,old_status,new_status,note,event_at
5001,1001,101,created,,open,Reported through mobile form,2026-02-01T23:10:00Z
5002,1001,201,assigned,open,open,Electrical crew notified,2026-02-02T14:05:00Z
5003,1002,102,created,,open,Pickup was scheduled for Monday,2026-02-02T15:45:00Z
5004,1002,202,status_changed,open,in_progress,Route supervisor checking vehicle log,2026-02-03T13:30:00Z
5005,1003,103,created,,open,Pressure lower on two floors,2026-02-03T12:05:00Z
5006,1003,201,status_changed,open,in_progress,Crew dispatched,2026-02-03T14:25:00Z
5007,1003,201,status_changed,in_progress,resolved,Valve adjustment restored pressure,2026-02-03T19:40:00Z
5008,1004,104,created,,new,Photo attached in original report,2026-02-04T17:20:00Z
5009,1005,101,created,,open,Bin blocks part of sidewalk,2026-02-05T14:00:00Z
5010,1005,202,status_changed,open,resolved,Extra collection completed,2026-02-05T20:15:00Z
5011,1006,102,created,,open,Flicker repeats every few seconds,2026-02-06T01:30:00Z
5012,1006,201,status_changed,open,in_progress,Ballast inspection scheduled,2026-02-06T15:10:00Z
5013,1007,103,created,,open,Gate opens toward play area,2026-02-07T16:10:00Z
5014,1008,104,created,,open,Small stream along curb,2026-02-08T10:25:00Z
5015,1008,201,status_changed,open,resolved,Gasket replaced and area checked,2026-02-09T09:05:00Z
5016,1009,101,created,,new,No sharp edge visible,2026-02-09T22:15:00Z
5017,1010,102,created,,open,Current bin lid is missing,2026-02-10T13:50:00Z
5018,1010,202,status_changed,open,closed,Replacement delivered,2026-02-12T16:30:00Z
5019,1011,103,created,,open,Wait exceeds one full light cycle,2026-02-11T08:35:00Z
5020,1012,104,created,,open,Possible photocell issue,2026-02-12T14:45:00Z
5021,1012,202,status_changed,open,resolved,Photocell cleaned and tested,2026-02-14T18:10:00Z
""",
}

tables = {name: list(csv.DictReader(io.StringIO(text)))
          for name, text in CSV_DATA.items()}
users, tickets, events = (tables[name] for name in ("users", "tickets", "ticket_events"))
print("Table counts:", {name: len(rows) for name, rows in tables.items()})
print("Ticket statuses:", dict(sorted(Counter(t["status"] for t in tickets).items())))
print("Available statuses:", sorted({t["status"] for t in tickets}))

Run with **STATUS = "open"** and record the ordered IDs. Then change it to **"new"**, run again, and record the new IDs. The code filters tickets and orders the matches newest first, with ticket ID as a tie-breaker. MAX_ROWS is a ceiling, not a promise that twenty matches exist.

In [ ]:
STATUS = "open"  # First run: open. Second run: change to new.
MAX_ROWS = 20

def opening_time(ticket):
    return datetime.fromisoformat(ticket["opened_at"].replace("Z", "+00:00"))

matching = [t for t in tickets if t["status"] == STATUS]
selected = sorted(matching,
                  key=lambda t: (opening_time(t), int(t["ticket_id"])),
                  reverse=True)[:MAX_ROWS]
selected_ids = [int(t["ticket_id"]) for t in selected]
print("Status:", STATUS, " Matching tickets:", len(matching))
print("Ordered IDs:", selected_ids)
for t in selected:
    print(t["ticket_id"], t["opened_at"], t["subject"])
print("These are Python data-inspection results, not a PostgreSQL query plan.")

Use the next cell to inspect ticket **1004** and its requester. You may change TICKET_ID to another ID you saw above. A blank assignee_id in the CSV means the ticket has no assigned agent. That does not mean it lacks a requester.

In [ ]:
TICKET_ID = 1004  # Try another existing ID if you have time.
user_by_id = {u["user_id"]: u for u in users}
ticket = next((t for t in tickets if int(t["ticket_id"]) == TICKET_ID), None)
if ticket is None:
    print("Choose an existing ticket ID from 1001 through 1012.")
else:
    print(json.dumps({"ticket_id": TICKET_ID,
                      "subject": ticket["subject"],
                      "priority": ticket["priority"],
                      "status": ticket["status"],
                      "requester": user_by_id[ticket["requester_id"]]["display_name"],
                      "assignee_id": ticket["assignee_id"] or "unassigned"}, indent=2))
    history = [e for e in events if int(e["ticket_id"]) == TICKET_ID]
    print("Recorded events for this ticket:", len(history))

### Your observations

Name and section: **[write here]**

| Observation | Your result |
|---|---|
| Users / tickets / events | [counts from the first cell] |
| Ordered open-ticket IDs | [record before changing STATUS] |
| Ordered new-ticket IDs | [record after changing STATUS] |
| One ticket's ID, status, and requester | [record the ID you inspected] |

Keep the output from your last run. This table preserves both status results.

## 2 Read two short sources

Allow about 10 minutes. Open both official PostgreSQL sources below. These links target version 15 so the material stays consistent with the course examples.

1. [PostgreSQL Indexes Introduction](https://www.postgresql.org/docs/15/indexes-intro.html). Read the opening example through the paragraph beginning “Once an index is created,” then the final paragraph about keeping an index synchronized with its table. Skip the operator-class discussion.
2. [PostgreSQL Indexes and ORDER BY](https://www.postgresql.org/docs/15/indexes-ordering.html). Read the first three paragraphs, through the discussion of ORDER BY with LIMIT. The later mixed-direction examples are optional.

Look for what an index might avoid, why PostgreSQL might choose another plan, and what work an index adds. Paraphrase the ideas you use and keep the links in your response. You do not need to search for more sources or copy definitions.

## 3 Write your preliminary plan

Allow about 25 minutes, including a quick reread. Write **180–250 words total** across the next three cells. Use your own wording. Connect documentation to the actual dataset and separate an observation from a prediction. This is a starting plan you can revise after running PostgreSQL tests.

The proposed PostgreSQL query and index in the starter use **status**, **opened_at DESC**, and **ticket_id DESC**. The last key breaks timestamp ties. You can refer to these columns in your plan; you do not need to write or execute new SQL today.

### A The question and data

Aim for 50–70 words. What does the newest-open-tickets query need to return? Use two specific observations from your notebook. Explain why LIMIT 20 can return fewer than twenty rows in this dataset.

**Your response:** [replace this prompt]

### B What the sources suggest

Aim for 80–110 words. Explain how an index could help the filter or ordering and name one cost. Use an idea from each source, linking it where you use it. Explain why finding an index definition online does not establish that this small database will run faster.

**Your response:** [replace this prompt and retain the two source links]

[Indexes Introduction](https://www.postgresql.org/docs/15/indexes-intro.html) · [Indexes and ORDER BY](https://www.postgresql.org/docs/15/indexes-ordering.html)

### C Your planned test and remaining question

Aim for 50–70 words. Describe one before-and-after test you will run in PostgreSQL for the midterm. State what stays the same, which result and plan evidence you will compare, and what would make you keep or reconsider the index. Label your expected outcome as a prediction. Finish with one question you still need answered.

**Your response:** [replace this prompt]

## Save and submit

Use the last five minutes of class to check that the observation table and all three responses are complete, your two sources are linked, and outputs are visible. If a runtime problem prevented execution, label any supplied example you discussed and identify the steps you could not run. Do not invent observations.

Run every cell, then choose **File > Save** so the outputs and writing are stored in your Drive copy. Choose **Share**, give the instructor viewer access, and copy the link. Submit that sharing link through today's classwork assignment in Brightspace. Submit your saved copy's link, not this starter's GitHub or Colab link. You do not need a downloaded file, separate document, or screenshots.

Reuse this plan in your final README after replacing predictions with your PostgreSQL results. This notebook is today's preparation, not the final midterm submission.

Original instructional content is licensed CC BY-NC-SA 4.0. Original code MIT. Metro Support synthetic data CC0. The linked PostgreSQL documentation retains its own license.